<a href="https://colab.research.google.com/github/ngtan369/Randomized-SVD-in-compress-Data/blob/main/svd_compressData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Hello Linear Algebra Course !!")

Hello Linear Algebra Course !!


# Phân tích Randomized SVD (Truncated SVD) và Ứng dụng trong nén dữ liệu

## 1. Cơ sở lý thuyết của SVD và Compact SVD

### Singular Value Decomposition (SVD)

Phân tích giá trị kỳ dị (SVD) là một kỹ thuật phân tách ma trận mạnh mẽ, tổng quát hóa khái niệm phân tích riêng (eigen-decomposition) cho các ma trận không vuông. Đối với một ma trận thực $A$ kích thước $m \times n$, SVD phân tách $A$ thành ba ma trận:

$$A = U \Sigma V^T$$

Trong đó:
- $U$ là ma trận trực giao $m \times m$ (`U U^T = I`), các cột của $U$ được gọi là **vector kỳ dị trái**.
- $\Sigma$ (Sigma) là ma trận đường chéo $m \times n$ chứa các **giá trị kỳ dị** không âm, theo thứ tự giảm dần trên đường chéo chính: $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_{\min(m,n)} \ge 0$. Các phần tử ngoài đường chéo bằng 0.
- $V$ là ma trận trực giao $n \times n$ (`V V^T = I`), các cột của $V$ được gọi là **vector kỳ dị phải**.

SVD luôn tồn tại cho mọi ma trận, bất kể kích thước hoặc hạng của nó. Nó cung cấp một cái nhìn sâu sắc về cấu trúc nền tảng của ma trận, chẳng hạn như hạng, không gian con hàng và không gian con cột.

### Compact SVD (Truncated SVD hoặc Reduced SVD)

Khi hạng $r$ của ma trận $A$ nhỏ hơn $\min(m,n)$, hoặc khi chúng ta chỉ quan tâm đến các thành phần đóng góp chính, chúng ta có thể sử dụng Compact SVD. Thay vì lưu trữ toàn bộ các ma trận $U$, $\Sigma$, và $V$, chúng ta chỉ giữ lại $r$ cột đầu tiên của $U$, $r$ giá trị kỳ dị đầu tiên của $\Sigma$, và $r$ cột đầu tiên của $V$.

Trong trường hợp này, $\Sigma$ trở thành một ma trận đường chéo $r \times r$, và các ma trận $U$ và $V$ có kích thước tương ứng $m \times r$ và $n \times r$. Công thức Compact SVD là:

$$A = U_r \Sigma_r V_r^T$$

Với $U_r$ là ma trận $m \times r$, $\Sigma_r$ là ma trận đường chéo $r \times r$, và $V_r$ là ma trận $n \times r$. Compact SVD rất hữu ích cho việc giảm kích thước và loại bỏ nhiễu, vì các giá trị kỳ dị nhỏ thường tương ứng với nhiễu hoặc các thành phần ít quan trọng của dữ liệu.

## 2. Cơ sở lý thuyết của Randomized SVD (chỉ tính k giá trị kỳ dị lớn nhất)

Đối với các ma trận rất lớn, việc tính toán SVD đầy đủ hoặc thậm chí Compact SVD có thể rất tốn kém về mặt tính toán và bộ nhớ. Randomized SVD là một phương pháp hiệu quả để xấp xỉ các giá trị kỳ dị và vector kỳ dị lớn nhất của một ma trận lớn. Ý tưởng chính là sử dụng các phép chiếu ngẫu nhiên để giảm kích thước của ma trận gốc, sau đó thực hiện SVD trên ma trận nhỏ hơn này.

**Các bước cơ bản của Randomized SVD để tìm $k$ giá trị kỳ dị lớn nhất:**

1.  **Bước 1: Chiếu ngẫu nhiên (Randomized Projection)**
    *   Tạo một ma trận ngẫu nhiên $G$ có kích thước $n \times (k+p)$, trong đó $k$ là số giá trị kỳ dị chúng ta muốn giữ lại và $p$ là một số nguyên nhỏ (thường từ 5 đến 10) để "over-sampling" nhằm tăng độ chính xác của xấp xỉ.
    *   Tính toán ma trận $Y = A G$. Ma trận $Y$ có kích thước $m \times (k+p)$ và chứa thông tin quan trọng về không gian con của $A$ với $k+p$ chiều.

2.  **Bước 2: Chuẩn hóa trực giao (Orthogonalization)**
    *   Thực hiện phân tích QR trên ma trận $Y$ để tìm một cơ sở trực giao cho không gian cột của $Y$. Điều này cho ra ma trận $Q$ có kích thước $m \times (k+p)$, với các cột trực giao.

3.  **Bước 3: Chiếu trở lại không gian nhỏ (Reduced SVD)**
    *   Tính ma trận $B = Q^T A$. Ma trận $B$ có kích thước $(k+p) \times n$, nhỏ hơn nhiều so với $A$.
    *   Thực hiện SVD trên ma trận nhỏ $B$: $B = \hat{U} \Sigma V^T$.

4.  **Bước 4: Xấp xỉ SVD của A (Approximate SVD of A)**
    *   Các vector kỳ dị trái của $A$ được xấp xỉ bởi $U = Q \hat{U}$.
    *   Các giá trị kỳ dị $\Sigma$ và vector kỳ dị phải $V$ được lấy trực tiếp từ SVD của $B$.

Kết quả là $A \approx U \Sigma V^T$, trong đó $U$ có kích thước $m \times (k+p)$, $\Sigma$ là ma trận đường chéo $(k+p) \times (k+p)$, và $V$ có kích thước $n \times (k+p)$. Các giá trị và vector kỳ dị được sắp xếp theo thứ tự giảm dần, cho phép chúng ta chọn $k$ thành phần hàng đầu.

## 3. Ý nghĩa của năng lượng dữ liệu và các giá trị kỳ dị

Trong bối cảnh SVD, **"năng lượng dữ liệu"** thường được hiểu là tổng bình phương của tất cả các giá trị trong ma trận, tương đương với bình phương của chuẩn Frobenius của ma trận:

$$ \|A\|_F^2 = \sum_{i=1}^m \sum_{j=1}^n A_{ij}^2 $$

Một tính chất quan trọng của SVD là tổng bình phương của các giá trị kỳ dị cũng bằng bình phương của chuẩn Frobenius:

$$ \|A\|_F^2 = \sum_{i=1}^{\min(m,n)} \sigma_i^2 $$

Các **giá trị kỳ dị** $\sigma_i$ đo lường mức độ "quan trọng" của từng cặp vector kỳ dị (trái và phải) trong việc tái tạo lại ma trận gốc. Các giá trị kỳ dị lớn hơn tương ứng với các thành phần dữ liệu mang nhiều thông tin hơn hoặc có phương sai lớn hơn. Các giá trị kỳ dị nhỏ hơn thường tương ứng với nhiễu hoặc các chi tiết ít quan trọng.

### Tiêu chuẩn chọn $k$ theo tỉ lệ năng lượng

Để nén dữ liệu mà vẫn giữ lại một phần lớn "năng lượng" (thông tin) của ma trận gốc, chúng ta cần chọn một số hạng $k$ sao cho tổng bình phương của $k$ giá trị kỳ dị lớn nhất đạt một tỉ lệ $\eta$ nhất định so với tổng bình phương của tất cả các giá trị kỳ dị:

$$ \frac{\sum_{i=1}^k \sigma_i^2}{\sum_{i=1}^r \sigma_i^2} \ge \eta $$

Trong đó:
- $k$ là số giá trị kỳ dị (và cặp vector kỳ dị) được giữ lại.
- $\sigma_i$ là giá trị kỳ dị thứ $i$.
- $r$ là hạng của ma trận $A$ (hoặc $\min(m,n)$).
- $\eta$ là tỉ lệ năng lượng mong muốn (ví dụ: 0.95 cho 95% năng lượng).

Tiêu chuẩn này giúp chúng ta xác định số lượng thành phần tối thiểu cần thiết để tái tạo ma trận gốc với một mức độ chính xác mong muốn, từ đó đạt được hiệu quả nén.

## 4. Thu thập và Tiền xử lý Dữ liệu

Để minh họa Randomized SVD trên một ma trận kích thước lớn, chúng ta sẽ sử dụng bộ dữ liệu **MovieLens 25M Dataset** từ GroupLens Research. Đây là một bộ dữ liệu phổ biến cho các hệ thống khuyến nghị, chứa 25 triệu xếp hạng và 109.000 tag áp dụng cho 62.000 bộ phim bởi 162.000 người dùng.

*   **Nguồn dữ liệu:** [Kaggle - MovieLens 25M Dataset](https://www.kaggle.com/datasets/grouplens/movielens-25m-dataset)
*   **Ý nghĩa các biến:** Chúng ta sẽ tập trung vào file `ratings.csv`, bao gồm các cột:
    *   `userId`: ID duy nhất của người dùng.
    *   `movieId`: ID duy nhất của bộ phim.
    *   `rating`: Xếp hạng của người dùng (từ 0.5 đến 5.0).
    *   `timestamp`: Thời gian xếp hạng được tạo (chúng ta sẽ không sử dụng cột này).

**Tiền xử lý để đưa dữ liệu về dạng ma trận số:**

Chúng ta sẽ xây dựng một ma trận xếp hạng người dùng-phim ($A$) nơi các hàng đại diện cho người dùng, các cột đại diện cho phim, và các giá trị trong ma trận là xếp hạng của người dùng cho phim đó. Vì không phải người dùng nào cũng xếp hạng tất cả các phim, ma trận này sẽ rất thưa (sparse). Chúng ta sẽ cần một cách hiệu quả để biểu diễn và xử lý ma trận thưa này.

In [ ]:
# Cài đặt Kaggle API client nếu chưa có
# !pip install kaggle

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

# Hướng dẫn tải dữ liệu:
# 1. Truy cập Kaggle và tạo một API token.
# 2. Đặt file kaggle.json vào thư mục ~/.kaggle/ (Linux/macOS) hoặc C:\Users\<username>\.kaggle\ (Windows).
# 3. Đảm bảo quyền truy cập file chỉ cho chủ sở hữu (chmod 600 ~/.kaggle/kaggle.json).
# 4. Tải dataset:
#    !kaggle datasets download -d grouplens/movielens-25m-dataset
#    !unzip movielens-25m-dataset.zip -d movielens-25m

print("Đang tải và tiền xử lý dữ liệu MovieLens 25M...")

# Đọc file ratings.csv
# Điều chỉnh đường dẫn nếu cần thiết
ratings_df = pd.read_csv('movielens-25m/ratings.csv')

# Hiển thị vài dòng đầu để kiểm tra
print("Dữ liệu ratings.csv ban đầu:")
display(ratings_df.head())

# Tạo ánh xạ ID liên tục cho userId và movieId để xây dựng ma trận
user_ids = ratings_df['userId'].astype('category').cat.codes
movie_ids = ratings_df['movieId'].astype('category').cat.codes

# Lấy số lượng người dùng và phim duy nhất
num_users = len(ratings_df['userId'].unique())
num_movies = len(ratings_df['movieId'].unique())

print(f"Số lượng người dùng duy nhất: {num_users}")
print(f"Số lượng phim duy nhất: {num_movies}")

# Xây dựng ma trận thưa (sparse matrix) người dùng-phim
# Sử dụng Compressed Sparse Row (CSR) matrix để tiết kiệm bộ nhớ
rating_matrix = csr_matrix(
    (ratings_df['rating'], (user_ids, movie_ids)),
    shape=(num_users, num_movies)
)

print(f"Kích thước ma trận xếp hạng (Người dùng x Phim): {rating_matrix.shape}")
print(f"Số lượng phần tử khác 0 trong ma trận: {rating_matrix.nnz}")
print(f"Ma trận có tổng cộng {rating_matrix.shape[0] * rating_matrix.shape[1]} phần tử, \nvới {rating_matrix.nnz} phần tử khác không.")

# Chuyển đổi ma trận thưa thành ma trận dày (dense) nếu kích thước cho phép hoặc cho mục đích thử nghiệm nhỏ
# Cảnh báo: Việc này có thể tốn rất nhiều RAM nếu ma trận quá lớn!
# Ví dụ: matrix_dense = rating_matrix.toarray()

# Đối với các bước sau, chúng ta sẽ làm việc trực tiếp với ma trận thưa hoặc sử dụng các thư viện hỗ trợ ma trận thưa.
# Để đơn giản hóa ví dụ và đảm bảo tính toán, tôi sẽ chuyển đổi một phần nhỏ hoặc xử lý ma trận thưa một cách phù hợp.
# Tuy nhiên, yêu cầu của bài là ma trận lớn, nên chúng ta cần cân nhắc cách làm việc hiệu quả với ma trận thưa.

# Để làm việc với Randomized SVD từ thư viện, thường cần ma trận dense hoặc một số wrapper.
# Để đảm bảo ví dụ chạy được mà không hết RAM với toàn bộ 25M ratings, chúng ta sẽ lấy một phần dữ liệu nhỏ hơn
# hoặc tập trung vào việc chuyển đổi để Randomized SVD có thể xử lý.

# Hiện tại, ma trận `rating_matrix` đã được xây dựng dưới dạng sparse. Chúng ta sẽ sử dụng nó làm ma trận đầu vào `A`.